### Time Series Analysis

**The Video, Not The Snapshot - Time Series Analysis For Business Value**

*Goal: Why Time-dependent data requires special treatment*

- why does time series analysis matter?

- A single photo(Regular Datasets) vs The Video(Time Series Data)

- Time series analysis treats data like a video

- Data having timestamps

- You can't use train-test-split(shuffle = True) on time series data

- A look-ahead-bias(Time travel data leakage)

**Three Components of Time Series Data**

- Trend(The long-term slope): Is the metric falling or rising over months/years?

- Seasonality(The Repeating Clock): The predictable pattern that repeats at regular intervals

- Autocorrelation(Today's Momentum): How today's value depends on yesterday's values

**Climate Risk & Temperature Forecasting**

*NOAA's Daily Weather Observation*

In [1]:
# Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

In [2]:
# Dataset of interest
# Loaded raw NOAA observations
pd.set_option('display.max_columns', None)
noaa_data = pd.read_csv('/content/drive/MyDrive/AltSchool_Africa/4th_Semester/noaa - 5000.csv')
noaa_data.head()


,stn,wban,date,year,mo,da,temp,count_temp,dewp,count_dewp,slp,count_slp,stp,count_stp,visib,count_visib,wdsp,count_wdsp,mxpsd,gust,max,flag_max,min,flag_min,prcp,flag_prcp,sndp,fog,rain_drizzle,snow_ice_pellets,hail,thunder,tornado_funnel_cloud
0,10020,99999,2025-07-04,2025,7,4,42.1,4,38.5,4,1004.4,4,3.2,4,999.9,0,7.6,4,9.7,999.9,42.6,NaN,41.2,NaN,0.0,I,999.9,0,0,0,0,0,0
1,10020,99999,2025-04-30,2025,4,30,7.6,4,3.8,4,1002.0,4,0.8,4,999.9,0,18.0,4,23.3,999.9,10.0,NaN,5.7,NaN,0.0,I,999.9,0,0,0,0,0,0
2,10020,99999,2025-04-19,2025,4,19,2.7,4,-2.5,4,1015.5,4,14.3,4,999.9,0,12.8,4,13.6,14.0,4.3,NaN,1.8,NaN,0.0,I,999.9,0,0,0,0,0,0
3,10020,99999,2025-04-20,2025,4,20,4.2,4,-1.0,4,1021.4,4,20.2,4,999.9,0,5.2,4,7.8,999.9,5.9,NaN,3.0,NaN,0.0,I,999.9,0,0,0,0,0,0
4,10020,99999,2025-03-05,2025,3,5,14.4,4,10.0,4,1003.0,4,1.8,4,999.9,0,18.9,4,21.4,999.9,18.7,NaN,9.5,NaN,0.0,I,999.9,0,0,0,0,0,0


In [3]:
noaa_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 33 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   stn                   5000 non-null   int64  
 1   wban                  5000 non-null   int64  
 2   date                  5000 non-null   object 
 3   year                  5000 non-null   int64  
 4   mo                    5000 non-null   int64  
 5   da                    5000 non-null   int64  
 6   temp                  5000 non-null   float64
 7   count_temp            5000 non-null   int64  
 8   dewp                  5000 non-null   float64
 9   count_dewp            5000 non-null   int64  
 10  slp                   5000 non-null   float64
 11  count_slp             5000 non-null   int64  
 12  stp                   5000 non-null   float64
 13  count_stp             5000 non-null   int64  
 14  visib                 5000 non-null   float64
 15  count_visib          

In [4]:
# convert the date column to a datetime object
noaa_data['date'] = pd.to_datetime(noaa_data['date'])

In [5]:
noaa_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 33 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   stn                   5000 non-null   int64         
 1   wban                  5000 non-null   int64         
 2   date                  5000 non-null   datetime64[ns]
 3   year                  5000 non-null   int64         
 4   mo                    5000 non-null   int64         
 5   da                    5000 non-null   int64         
 6   temp                  5000 non-null   float64       
 7   count_temp            5000 non-null   int64         
 8   dewp                  5000 non-null   float64       
 9   count_dewp            5000 non-null   int64         
 10  slp                   5000 non-null   float64       
 11  count_slp             5000 non-null   int64         
 12  stp                   5000 non-null   float64       
 13  count_stp         

In [6]:
# Aggregate multiple station readings
daily = noaa_data.groupby('date')['temp'].mean().reset_index().sort_values('date')
daily.head()

,date,temp
0,2025-01-01,50.387500
1,2025-01-02,45.115000
2,2025-01-03,48.352632
3,2025-01-04,43.564286
4,2025-01-05,47.028571


In [7]:
# set 'date' as the explicit Time Index and ensure daily frequency
daily = daily.set_index('date').asfreq('D')

In [8]:
daily.head()

,temp
date,
2025-01-01,50.387500
2025-01-02,45.115000
2025-01-03,48.352632
2025-01-04,43.564286
2025-01-05,47.028571


In [9]:
# DEal with missing values
daily['temp'] = daily['temp'].interpolate(method = 'linear')

In [10]:
print(f'Time Series Range: {daily.index.min().strftime('%Y-%m-%d')} to {daily.index.max().strftime('%Y-%m-%d')}')
print(f'Total Consecutive Days: {len(daily)}')
#

Time Series Range: 2025-01-01 to 2025-08-28
Total Consecutive Days: 240


In [11]:
# Time Series Feature Engineering
# Lag Features
daily['temp_lag_1'] = daily['temp'].shift(1) # Yesterday
daily['temp_lag_2'] = daily['temp'].shift(2) # 2 Days Ago
daily['temp_lag_7'] = daily['temp'].shift(7) # 7 Days Ago

In [12]:
daily.head(20)

,temp,temp_lag_1,temp_lag_2,temp_lag_7
date,,,,
2025-01-01,50.387500,NaN,NaN,NaN
2025-01-02,45.115000,50.387500,NaN,NaN
2025-01-03,48.352632,45.115000,50.387500,NaN
2025-01-04,43.564286,48.352632,45.115000,NaN
2025-01-05,47.028571,43.564286,48.352632,NaN
2025-01-06,42.575000,47.028571,43.564286,NaN
2025-01-07,38.847619,42.575000,47.028571,NaN
2025-01-08,42.275000,38.847619,42.575000,50.387500
2025-01-09,47.921053,42.275000,38.847619,45.115000


In [13]:
# Rolling Window Feature: What was the average temperature over the past week
daily['temp_rolling_7'] = daily['temp'].shift(1).rolling(window= 7).mean()

In [14]:
daily.head(20)

,temp,temp_lag_1,temp_lag_2,temp_lag_7,temp_rolling_7
date,,,,,
2025-01-01,50.387500,NaN,NaN,NaN,NaN
2025-01-02,45.115000,50.387500,NaN,NaN,NaN
2025-01-03,48.352632,45.115000,50.387500,NaN,NaN
2025-01-04,43.564286,48.352632,45.115000,NaN,NaN
2025-01-05,47.028571,43.564286,48.352632,NaN,NaN
2025-01-06,42.575000,47.028571,43.564286,NaN,NaN
2025-01-07,38.847619,42.575000,47.028571,NaN,NaN
2025-01-08,42.275000,38.847619,42.575000,50.387500,45.124373
2025-01-09,47.921053,42.275000,38.847619,45.115000,43.965444


In [15]:
# Calender Features: Capture annual Seasonality
daily['month'] = daily.index.month
daily['day_of_year'] = daily.index.dayofyear

In [16]:
daily.head(20)

,temp,temp_lag_1,temp_lag_2,temp_lag_7,temp_rolling_7,month,day_of_year
date,,,,,,,
2025-01-01,50.387500,NaN,NaN,NaN,NaN,1,1
2025-01-02,45.115000,50.387500,NaN,NaN,NaN,1,2
2025-01-03,48.352632,45.115000,50.387500,NaN,NaN,1,3
2025-01-04,43.564286,48.352632,45.115000,NaN,NaN,1,4
2025-01-05,47.028571,43.564286,48.352632,NaN,NaN,1,5
2025-01-06,42.575000,47.028571,43.564286,NaN,NaN,1,6
2025-01-07,38.847619,42.575000,47.028571,NaN,NaN,1,7
2025-01-08,42.275000,38.847619,42.575000,50.387500,45.124373,1,8
2025-01-09,47.921053,42.275000,38.847619,45.115000,43.965444,1,9


In [17]:
# drop the NaNs cused by shifting/lagging
clean_daily = daily.dropna().copy()
clean_daily.head()

,temp,temp_lag_1,temp_lag_2,temp_lag_7,temp_rolling_7,month,day_of_year
date,,,,,,,
2025-01-08,42.275000,38.847619,42.575000,50.387500,45.124373,1,8
2025-01-09,47.921053,42.275000,38.847619,45.115000,43.965444,1,9
2025-01-10,50.200000,47.921053,42.275000,48.352632,44.366309,1,10
2025-01-11,47.033333,50.200000,47.921053,43.564286,44.630218,1,11
2025-01-12,53.018182,47.033333,50.200000,47.028571,45.125797,1,12


In [17]:
# Chronological Time Split